###Paths

In [0]:
from pyspark.sql import functions as F

BRONZE_PATH = "abfss://bronze@ecommercedatalakebp01.dfs.core.windows.net"
SILVER_PATH = "abfss://silver@ecommercedatalakebp01.dfs.core.windows.net"
INGESTION_DATE = "2026-09-23"

In [0]:
path_customers=f"{BRONZE_PATH}/customers/ingestion_date={INGESTION_DATE}/customers.parquet"
df_customers=spark.read.parquet(path_customers)
df_customers.display()
df_customers.count()

In [0]:
df_customers_clean=df_customers.fillna({"email":"Unknown", "city":"unknown"})
print("Null emails :",df_customers_clean.filter(df_customers_clean.email.isNull()).count())
print("Null cities :",df_customers_clean.filter(df_customers_clean.city.isNull()).count())


In [0]:
def count_nulls(df):
    return df.select([F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in df.columns]).display()
count_nulls(df_customers_clean)

In [0]:
df_customers_clean=df_customers_clean.withColumn("signup_date", F.to_date(df_customers_clean.signup_date, "yyyy-MM-dd"))
df_customers_clean.printSchema()

In [0]:
df_customers_silver=(
    df_customers_clean
    .withColumn("_ingestion_date", F.lit(INGESTION_DATE))
    .withColumn("_proccessed_at",F.current_timestamp()))
df_customers_silver.display()


In [0]:
df_customers_silver.select("customer_id").distinct().count()

In [0]:
assert df_customers_silver.select("customer_id").distinct().count() == df_customers_silver.count(), "Duplicate customer_id found!"

In [0]:
customers_silver_path=f"{SILVER_PATH}/customers"
(df_customers_silver.write
 .mode("overwrite")
 .format("delta")
 .save(customers_silver_path))

In [0]:
spark.sql("DROP TABLE IF EXISTS customers_silver_path")

In [0]:
check = spark.read.format("delta").load(customers_silver_path)
print("rows in silver:", check.count())


In [0]:
products_path = f"{BRONZE_PATH}/products/ingestion_date={INGESTION_DATE}/products.csv"
df_products = spark.read.format("csv").option("header", "True").load(products_path)
df_products.display()
df_products.count()
df_products.printSchema()

In [0]:
df_products_clean=df_products.withColumn("price",F.col("price").cast("decimal(10,2)"))
df_products_clean.printSchema()


In [0]:
count_nulls(df_products_clean)

In [0]:
df_products_fixed=(df_products_clean
.withColumn("price_sign_corrected", F.when(F.col("price") < 0,True).otherwise(False))
.withColumn("price",F.abs(F.col("price"))))

In [0]:
df_products_fixed.filter(F.col("price")<0).count()

In [0]:
df_products_fixed.filter(F.col("price_sign_corrected") == True).count()

In [0]:
df_products_silver=(df_products_fixed
                    .withColumn("_ingestion_date", F.lit(INGESTION_DATE))
                    .withColumn("_proccessed_at",F.current_timestamp()))
df_products_silver.display()

In [0]:
assert df_products_silver.select("product_id").distinct().count() == df_products_silver.count(),"Duplicates have been found!"

In [0]:
products_silver_path=f"{SILVER_PATH}/products"
(df_products_silver.write.mode("overwrite").format("delta").save(products_silver_path))


In [0]:
print("rows in silver:", spark.read.format("delta").load(products_silver_path).count())

###Silver:Orders

In [0]:
orders_path=f"{BRONZE_PATH}/orders/ingestion_date={INGESTION_DATE}/"
df_orders=spark.read.format("parquet").load(orders_path)
df_orders.display()
df_orders.printSchema()
df_orders.count()


In [0]:
count_nulls(df_orders)

In [0]:
df_orders.select("order_id").distinct().count()

In [0]:
print("total rows:", df_orders.count())
print("distinct whole rows:", df_orders.distinct().count())

In [0]:
df_orders_fixed=df_orders.dropDuplicates()

In [0]:
df_orders_fixed.count()

In [0]:
df_orders_fixed=df_orders_fixed.fillna({"customer_id":"UNKNOWN"})

In [0]:
count_nulls(df_orders_fixed)

In [0]:
df_orders_fixed=(df_orders_fixed
                 .withColumn("order_date", F.to_date(df_orders_fixed.order_date,"yyyy-MM-dd"))
                 .withColumn("price", F.col("price").cast("decimal(10,2)"))
                 .withColumn("revenue", F.col("revenue").cast("decimal(12,2)")))
df_orders_fixed.display()
df_orders_fixed.printSchema()


In [0]:
count_nulls(df_orders_fixed)

In [0]:
df_orders_fixed=(df_orders_fixed
                 .withColumn("amount_sign_corrected", F.when((F.col("price")<0) | (F.col("revenue")<0), True).otherwise(False))
                 .withColumn("price", F.abs(F.col("price")))
                 .withColumn("revenue", F.abs(F.col("revenue"))))
df_orders_fixed.display()
df_orders_fixed.printSchema()


In [0]:
print("negative rows left:", df_orders_fixed.filter((F.col("price") < 0) | (F.col("revenue") < 0)).count())
print("orders corrected:", df_orders_fixed.filter(F.col("amount_sign_corrected")).count())

In [0]:
assert df_orders_fixed.select("order_id").distinct().count() == df_orders_fixed.count(),"Duplicates are found!"

In [0]:
df_orders_silver=(df_orders_fixed
                  .withColumn("_ingestion_date", F.to_date(F.lit(INGESTION_DATE)))
                  .withColumn("_processed_at", F.current_timestamp()))
df_orders_silver.display()
df_orders_silver.printSchema()


In [0]:
orders_silver_path = f"{SILVER_PATH}/orders"
df_orders_silver.write.mode("overwrite").format("delta").save(orders_silver_path)

In [0]:
print("rows in silver:", spark.read.format("delta").load(orders_silver_path).count())